# <center>Big Data &ndash; Exercise 1</center>
## <center>Fall 2026 &ndash; Week 1 &ndash; ETH Zurich</center>
## <center>Notebook for the SQL Moodle Quiz</center>

## Start Docker

Use the same ExamMagicBox and PostgreSQL database as in `Exercise01_SQL_Brush_Up.ipynb`.

1. Copy this notebook into the `notebooks` folder of your ExamMagicBox.
2. From the ExamMagicBox folder containing the Compose file, start the containers with `docker compose up -d`.
3. On the first startup, wait for the database import to finish. You can follow the database logs with `docker compose logs -f db`; wait for `PostgreSQL init process complete; ready for start up` and for PostgreSQL to accept connections.
4. Open Jupyter using the URL shown by the container and open this notebook with the Python 3 kernel.

When you are done, run `docker compose down` from the ExamMagicBox folder.

## Connect to PostgreSQL

Run the following cells in order to enable `%sql` and `%%sql`, connect to the database, and check the connection. The hostname `db` is available from inside the ExamMagicBox.

In [1]:
%reload_ext sql
%sql postgresql://postgres:example@db

Connecting to 'postgresql://postgres:***@db'

In [2]:
%%sql
SELECT version();

Running query in 'postgresql://postgres:***@db'

1 rows affected.

version
"PostgreSQL 17.6 (Debian 17.6-2.pgdg13+1) on aarch64-unknown-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit"


In [3]:
# Set this to the number of rows you want to display.
%config SqlMagic.displaylimit = 25

## The Moodle Quiz

**Week 1.2: SQL Brush-Up**

Use the Discogs dataset already loaded in the ExamMagicBox, as in the SQL brush-up exercise.

For each Moodle question, submit:

- The SQL query you wrote.
- The requested result of the query, following the format specified in Moodle.

You do not need to submit this notebook. Write each query in a code cell starting with `%%sql` and run it to obtain the result.

## Explore the dataset

The four Discogs tables are:

| Table | Contents | Join columns |
| --- | --- | --- |
| `artists` | Artists and their names | `artist_id` |
| `released_by` | Links between releases and artists | `release_id`, `artist_id` |
| `releases` | Release titles, dates, countries, and genres | `release_id` |
| `tracks` | Track titles, positions, and durations in seconds | `release_id` |

Run the following query to inspect the columns. A release can have multiple artists and multiple tracks; keep this in mind when counting after a join.

In [4]:
%%sql
SELECT table_name, column_name, data_type, is_nullable, ordinal_position
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name IN ('artists', 'released_by', 'releases', 'tracks')
ORDER BY table_name, ordinal_position;

Running query in 'postgresql://postgres:***@db'

17 rows affected.

table_name,column_name,data_type,is_nullable,ordinal_position
artists,artist_id,integer,NO,1
artists,name,character varying,YES,2
artists,realname,text,YES,3
artists,profile,text,YES,4
artists,url,text,YES,5
released_by,release_id,integer,NO,1
released_by,artist_id,integer,NO,2
releases,release_id,integer,NO,1
releases,released,date,NO,2
releases,title,text,NO,3


For example, the following query returns five artists. Use `LIMIT` while exploring to keep the output small.

In [5]:
%%sql
SELECT artist_id, name
FROM artists
ORDER BY artist_id
LIMIT 5;

Running query in 'postgresql://postgres:***@db'

5 rows affected.

artist_id,name
1,The Persuader
2,Mr. James Barth & A.D.
3,Josh Wink
4,Johannes Heil
5,Heiko Laux


## Question 1

Use an SQL query to find the number of artists that have at least one release with the genre 'Jazz' AND at least one track with a duration longer than 3600 seconds.

**Hints:**

- Note that 'Various Artists' should not count as an artist.
- The Jazz release and the long track do not need to belong to the same release.
- Duration is stored in seconds.
- The result must be given as an integer.
- Longer means strictly longer. 3600 is not longer than 3600.

In [6]:
%%sql
WITH artists_cleaned AS (
    SELECT * 
    FROM artists
    WHERE name != 'Various Artists'
),
at_least_one_release_jazz AS (
    SELECT a.artist_id
    FROM artists_cleaned a
    JOIN released_by rb USING(artist_id)
    JOIN releases r USING(release_id)
    WHERE r.genre = 'Jazz'
    GROUP BY a.artist_id
    HAVING COUNT(DISTINCT rb.release_id) >= 1
),
at_least_one_release_duration AS (
    SELECT a.artist_id
    FROM artists_cleaned a
    JOIN released_by rb USING(artist_id)
    WHERE EXISTS(
        SELECT *
        FROM tracks t
        WHERE t.release_id = rb.release_id
        AND t.duration > 3600
    )
    GROUP BY a.artist_id
)
SELECT COUNT(*) 
FROM (
    SELECT a.artist_id
    FROM at_least_one_release_jazz a
    INTERSECT
    SELECT a.artist_id
    FROM at_least_one_release_duration a
);

Running query in 'postgresql://postgres:***@db'

1 rows affected.

count
22


## Question 2

Use an SQL query to find the name of the artist who has the single longest track (by duration) in the dataset.

**Hints:**

- Duration is stored in seconds. Ignore tracks with a NULL duration.
- The solution is unique. Give the name exactly as it appears in the database.

In [12]:
%%sql
WITH longest_track AS (
    SELECT * FROM tracks
    WHERE duration IS NOT NULL
    ORDER BY duration DESC
    LIMIT 1
)
SELECT a.name
FROM artists a
JOIN released_by rb USING(artist_id)
JOIN longest_track lt USING(release_id);

Running query in 'postgresql://postgres:***@db'

1 rows affected.

name
Hair Stylistics
